In [1]:
# Cell 1: Environment Setup & Spark Initialization

In [2]:
# Imports
import os
import re
import time
import random
from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vectors

# Initialize Spark Context (local mode for development)
conf = SparkConf().setAppName("CSL7110_Assignment4").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf)
sc.setLogLevel("WARN")

print(" Spark Context initialized successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/19 22:28:08 WARN Utils: Your hostname, suvendu2023, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/19 22:28:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 22:28:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


 Spark Context initialized successfully.


In [25]:
import os

# Updated path to match your folder structure
DATA_DIR = "data/Q1-UCI-Spam-clustering1"
DATA_PATH = os.path.join(DATA_DIR, "spambase.data")  # Actual filename

# Verify dataset exists
if os.path.exists(DATA_PATH):
    print(f" Dataset found at: {DATA_PATH}")
else:
    print(f" Dataset not found at: {DATA_PATH}")
    print(f" Please ensure the file 'spambase.data' exists in the folder '{DATA_DIR}'")

# Quick validation
with open(DATA_PATH, 'r') as f:
    first_line = f.readline().strip()
    cols = len(first_line.split(','))
    
print(f" Verified: {cols} columns per row.")
print(f" First line preview: {first_line[:100]}...")
print(" Ready for readVectorsSeq().")

 Dataset found at: data/Q1-UCI-Spam-clustering1/spambase.data
 Verified: 58 columns per row.
 First line preview: 0,0.64,0.64,0,0.32,0,0,0,0,0,0,0.64,0,0,0,0.32,0,1.29,1.93,0,0.96,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...
 Ready for readVectorsSeq().


In [4]:
#  Cell 2: Part 1 - Clustering Functions

In [26]:
# CELL 2: Part 1 - Clustering Functions (Fixed for PySpark 3.x+)
from pyspark.mllib.linalg import Vectors
import random

def sq_dist(a, b):
    """Pure Python squared L2-distance. Works reliably on all PySpark versions."""
    return sum((x - y) ** 2 for x, y in zip(a, b))

def readVectorsSeq(filename):
    """Reads CSV file and returns a list of MLlib Vectors."""
    lines = sc.textFile(filename)
    return lines.map(lambda line: Vectors.dense([float(x) for x in line.strip().split(',')])).collect()

def kcenter(P, k):
    """
    Farthest-First Traversal. Returns k centers.
    Complexity: O(|P| * k) -> k iterations, each scans P once via RDD.map()
    """
    if k <= 0: return []
    rdd = sc.parallelize(P)
    centers = [rdd.first()]
    
    for _ in range(k - 1):
        centers_b = sc.broadcast(centers)
        # Compute min squared distance to any center for each point
        min_sq_dists = rdd.map(lambda p: min(sq_dist(p, c) for c in centers_b.value))
        # Pick point with maximum min-distance
        farthest = rdd.zip(min_sq_dists).max(lambda x: x[1])[0]
        centers.append(farthest)
    return centers

def kmeansPP(P, k):
    """
    k-means++ initialization. Returns k centers.
    Complexity: O(|P| * k) -> driver-side weighted sampling after O(|P|) map
    """
    if k <= 0: return []
    rdd = sc.parallelize(P)
    centers = [random.choice(P)]
    
    for _ in range(k - 1):
        centers_b = sc.broadcast(centers)
        # Compute D(x)^2 for all points
        sq_dists = rdd.map(lambda p: min(sq_dist(p, c) for c in centers_b.value)).collect()
        # Sample proportionally to D(x)^2
        next_c = random.choices(P, weights=sq_dists, k=1)[0]
        centers.append(next_c)
    return centers

def kmeansObj(P, C):
    """Returns average squared distance of P to closest center in C."""
    if not C: return 0.0
    rdd = sc.parallelize(P)
    c_b = sc.broadcast(C)
    avg = rdd.map(lambda p: min(sq_dist(p, c) for c in c_b.value)).mean()
    return avg

In [27]:
# 🔹 Cell 3: Part 1 - Execution & Testing

In [28]:
# Configuration
DATA_PATH = "data/Q1-UCI-Spam-clustering1/spambase.data"  # Updated path
K = 10
K1 = 50

print(" Loading dataset...")
P = readVectorsSeq(DATA_PATH)
print(f" Loaded {len(P)} points with {P[0].size} dimensions.\n")

# Validation
assert len(P) == 4601, f"Expected 4601 points, got {len(P)}"
assert P[0].size == 58, f"Expected 58 dimensions, got {P[0].size}"
print(" Dataset dimensions match assignment requirements (4601 × 58).")



 Loading dataset...


[Stage 2085:>                                                       (0 + 2) / 2]

 Loaded 4601 points with 58 dimensions.

 Dataset dimensions match assignment requirements (4601 × 58).


In [29]:
print(f" Type of P: {type(P)}")
print(f" Number of points: {len(P)}")
print(f" Dimensions per point: {P[0].size}")
print(f" Type of a single point: {type(P[0])}")
# Print first 2 points as readable arrays
for i in range(2):
    print(f"Point {i}: {P[i].toArray()}")

import numpy as np
import pandas as pd

# Convert list of Vectors -> 2D numpy array
data_array = np.array([v.toArray() for v in P])

print(" Shape:", data_array.shape)
print("\n First 5 rows:\n", data_array[:5])
print("\n Basic Statistics:")
print(pd.DataFrame(data_array).describe())

print(" Contains NaNs?", np.isnan(data_array).any())
print(" Global Min:", data_array.min())
print(" Global Max:", data_array.max())
print(" Last column unique values (class labels):", np.unique(data_array[:, -1]))

 Type of P: <class 'list'>
 Number of points: 4601
 Dimensions per point: 58
 Type of a single point: <class 'pyspark.mllib.linalg.DenseVector'>
Point 0: [  0.      0.64    0.64    0.      0.32    0.      0.      0.      0.
   0.      0.      0.64    0.      0.      0.      0.32    0.      1.29
   1.93    0.      0.96    0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.778   0.      0.
   3.756  61.    278.      1.   ]
Point 1: [2.100e-01 2.800e-01 5.000e-01 0.000e+00 1.400e-01 2.800e-01 2.100e-01
 7.000e-02 0.000e+00 9.400e-01 2.100e-01 7.900e-01 6.500e-01 2.100e-01
 1.400e-01 1.400e-01 7.000e-02 2.800e-01 3.470e+00 0.000e+00 1.590e+00
 0.000e+00 4.300e-01 4.300e-01 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 7.000e-02 0.000

In [30]:
# 1. Run kcenter(P, k)
start = time.time()
C_kc = kcenter(P, K)
print(f" kcenter(P, {K}) runtime: {time.time() - start:.4f} sec\n")


 kcenter(P, 10) runtime: 16.9786 sec



In [31]:
# 2. Run kmeansPP(P, k) + objective
C_pp = kmeansPP(P, K)
obj1 = kmeansObj(P, C_pp)
print(f" kmeansPP(P, {K}) objective: {obj1:.6f}\n")



 kmeansPP(P, 10) objective: 31249.248138



In [32]:
# 3. Coreset approach: kcenter(P, k1) -> X, then kmeansPP(X, k) -> C
X = kcenter(P, K1)
C_core = kmeansPP(X, K)
obj2 = kmeansObj(P, C_core)
print(f" kcenter(P, {K1}) -> kmeansPP(X, {K}) objective: {obj2:.6f}")
print(" Lower objective indicates better cluster centers.")

 kcenter(P, 50) -> kmeansPP(X, 10) objective: 420864.869463
 Lower objective indicates better cluster centers.


In [13]:
#

In [14]:
#🔹 Cell 4: Part 2 - Web Search Data Structures

In [15]:
# --- Constants ---
STOP_WORDS = {'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'}
PUNCTUATION = "{}[]<>=().,;'\"?!-:"
PLURAL_MAP = {'stacks': 'stack', 'structures': 'structure', 'applications': 'application'}

class MySet:
    def __init__(self): self.elements = set()
    def addElement(self, e): self.elements.add(e)
    def union(self, other):
        res = MySet(); res.elements = self.elements | other.elements; return res
    def intersection(self, other):
        res = MySet(); res.elements = self.elements & other.elements; return res

class Position:
    def __init__(self, p, wordIndex): self.page = p; self.wordIndex = wordIndex
    def getPageEntry(self): return self.page
    def getWordIndex(self): return self.wordIndex

class WordEntry:
    def __init__(self, word): self.word = word; self.positions = []
    def addPosition(self, p): self.positions.append(p)
    def addPositions(self, ps): self.positions.extend(ps)
    def getAllPositionsForThisWord(self): return self.positions
    def getTermFrequency(self, word=None): return len(self.positions)

class PageIndex:
    def __init__(self): self.table = {}
    def getHashIndex(self, s): return hash(s)
    def addPositionsForWord(self, w: WordEntry):
        if w.word not in self.table: self.table[w.word] = WordEntry(w.word)
        self.table[w.word].addPositions(w.getAllPositionsForThisWord())
    def getWordEntries(self): return list(self.table.values())
    def getEntry(self, word): return self.table.get(word)

class PageEntry:
    def __init__(self, pageName):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        self._processFile(f"data/Q2-webSearch/webpages/{pageName}")
    
    def _processFile(self, filepath):
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Webpage file not found: {filepath}")
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        for char in PUNCTUATION: text = text.replace(char, ' ')
        idx = 0
        for raw in text.split():
            word = raw.lower()
            word = PLURAL_MAP.get(word, word)
            if word not in STOP_WORDS:
                pos = Position(self, idx)
                entry = self.pageIndex.getEntry(word) or WordEntry(word)
                entry.addPosition(pos)
                self.pageIndex.addPositionsForWord(entry)
            idx += 1
            
    def getPageIndex(self): return self.pageIndex

class InvertedPageIndex:
    def __init__(self): self.index = PageIndex() # Reusing PageIndex as hash table
    def addPage(self, p: PageEntry):
        for we in p.getPageIndex().getWordEntries():
            self.index.addPositionsForWord(we)
    def getPagesWhichContainWord(self, word):
        res = MySet()
        entry = self.index.getEntry(word)
        if entry:
            for pos in entry.getAllPositionsForThisWord():
                res.addElement(pos.getPageEntry())
        return res



class SearchEngine:
    def __init__(self):
        self.invIndex = InvertedPageIndex()
        self.pages_db = {}
        
    def performAction(self, actionMsg):
        parts = actionMsg.strip().split()
        if not parts: return
        cmd = parts[0]
        
        if cmd == "addPage":
            name = parts[1]
            self.pages_db[name] = PageEntry(name)
            self.invIndex.addPage(self.pages_db[name])
            print(f" Added: {name}")
            
        elif cmd == "queryFindPagesWhichContainWord":
            word = parts[1].lower()
            pages = self.invIndex.getPagesWhichContainWord(word)
            if not pages.elements:
                print(f"No webpage contains word {word}")
            else:
                print(",".join(sorted([p.pageName for p in pages.elements])))
                
        elif cmd == "queryFindPositionsOfWordInAPage":
            word, page = parts[1].lower(), parts[2]
            if page not in self.pages_db:
                print(f"No webpage {page} found")
                return
            entry = self.pages_db[page].getPageIndex().getEntry(word)
            if not entry:
                print(f"Webpage {page} does not contain word {word}")
            else:
                idxs = sorted([p.getWordIndex() for p in entry.getAllPositionsForThisWord()])
                print(",".join(map(str, idxs)))

In [16]:
#🔹 Cell 5: Part 2 - Execution (Reads actions.txt)

In [17]:
engine = SearchEngine()

if os.path.exists("data/Q2-webSearch/actions.txt"):
    with open("data/Q2-webSearch/actions.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                print(f"\n Query: {line}")
                engine.performAction(line)
else:
    print(" actions.txt not found. Place it in data/actions.txt")


 Query: addPage stack_datastructure_wiki
 Added: stack_datastructure_wiki

 Query: queryFindPagesWhichContainWord delhi
No webpage contains word delhi

 Query: queryFindPagesWhichContainWord stack
stack_datastructure_wiki

 Query: queryFindPagesWhichContainWord wikipedia
stack_datastructure_wiki

 Query: queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines

 Query: queryFindPagesWhichContainWord allain
No webpage contains word allain

 Query: addPage stack_cprogramming
 Added: stack_cprogramming

 Query: queryFindPagesWhichContainWord allain
stack_cprogramming

 Query: queryFindPagesWhichContainWord C
stack_cprogramming

 Query: queryFindPagesWhichContainWord C++
stack_cprogramming

 Query: addPage stack_oracle
 Added: stack_oracle

 Query: queryFindPagesWhichContainWord jdk
stack_oracle

 Query: addPage stackoverflow
 Added: stackoverflow

 Query: queryFindPagesWhichContainWord function
stack_cprogramming,

In [18]:
#Cell 6: Part 3 - PageRank Implementation

In [19]:
def run_pagerank(input_file, beta=0.8, iterations=40):
    raw = sc.textFile(input_file)
    # Parse edges, remove duplicates
    edges = raw.map(lambda l: l.strip().split()).map(lambda x: (int(x[0]), int(x[1]))).distinct()
    
    # All unique nodes
    all_nodes = edges.flatMap(lambda x: [x[0], x[1]]).distinct()
    n = all_nodes.count()
    print(f" Graph: {n} nodes")
    
    # Out-degrees
    out_deg = edges.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b).collectAsMap()
    
    # Adjacency: (src, [(dst, weight)])
    adj = edges.map(lambda x: (x[0], (x[1], 1.0 / out_deg[x[0]])))
    adj_rdd = adj.groupByKey().mapValues(list)
    
    # Init ranks uniformly
    ranks = all_nodes.map(lambda node: (node, 1.0 / n))
    teleport = (1 - beta) / n
    
    for i in range(iterations):
        # r.join(adj) -> (src, (rank, [(dst, w)]))
        contrib = ranks.join(adj_rdd).flatMap(
            lambda src_data: [(dst, src_data[1][0] * w) for dst, w in src_data[1][1]]
        )
        new_ranks = contrib.reduceByKey(lambda a, b: a + b)
        # Ensure all nodes exist
        new_ranks = new_ranks.union(all_nodes.map(lambda x: (x, 0.0))).reduceByKey(lambda a, b: a + b)
        ranks = new_ranks.mapValues(lambda v: teleport + beta * v)
        
    # Results
    top5 = ranks.sortBy(lambda x: x[1], ascending=False).take(5)
    bottom5 = ranks.sortBy(lambda x: x[1], ascending=True).take(5)
    return top5, bottom5




In [20]:
# Validate small.txt (should have max PageRank ~0.036)
with open("graph/graphs/small.txt", "r") as f:
    edges = [line.strip().split() for line in f if line.strip()]
print(f" small.txt: {len(edges)} edges")
print(f" Sample edge: {edges[0]}")

# Validate whole.txt structure  
with open("graph/graphs//whole.txt", "r") as f:
    first_5 = [f.readline().strip() for _ in range(5)]
print(f" whole.txt first 5 lines:\n" + "\n".join(first_5))

 small.txt: 1024 edges
 Sample edge: ['100', '1']
 whole.txt first 5 lines:
1	2
2	3
3	4
4	5
5	6


In [21]:
#🔹 Cell 7: Part 3 - Execution

In [22]:
print(" Testing on small.txt...")
top_s, bot_s = run_pagerank("graph/graphs/small.txt", beta=0.8, iterations=40)
print("Top 5:", top_s)
print("Bottom 5:", bot_s)

print("\n Testing on whole.txt...")
top_l, bot_l = run_pagerank("graph/graphs/whole.txt", beta=0.8, iterations=40)
print("Top 5:", top_l)
print("Bottom 5:", bot_l)

 Testing on small.txt...


 Graph: 100 nodes


Top 5: [(53, 0.03573120223267159), (14, 0.03417090697259137), (40, 0.03363008718974389), (1, 0.0300059794797886), (27, 0.02972014420140538)]
Bottom 5: [(85, 0.003409694077402821), (59, 0.003669860660127284), (81, 0.0036953517493609907), (37, 0.0038082042916114506), (89, 0.003922466019802268)]

 Testing on whole.txt...
 Graph: 1000 nodes


[Stage 2083:=============================================>     (145 + 16) / 162]

Top 5: [(263, 0.0020202911815182184), (537, 0.0019433415714531497), (965, 0.0019254478071662631), (243, 0.001852634016241731), (285, 0.0018273721700645144)]
Bottom 5: [(558, 0.0003286018525215297), (93, 0.00035135689375165774), (62, 0.00035314810510596274), (424, 0.00035481538649301454), (408, 0.00038779848719291705)]
